In [ ]:
# Install YOLOv8 if not installed
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 65.2 MB/s eta 0:00:00


In [ ]:
# -------------------------------------------------------------
# 2-Quadrant Based Rotational Speed Detection
# -------------------------------------------------------------

import json
import time
from pathlib import Path
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from ultralytics.utils import LOGGER

LOGGER.setLevel("WARNING")


# =============================================================
# Speed Monitor
# =============================================================
class SpeedMonitor:

    def __init__(self):

        # ---------------- Model / Video Paths ----------------
        self.drum_segmentation_model_path = "model1.pt"
        self.marker_detection_model_path = "model2.pt"
        self.input_video_path = "video.mp4"

        # ---------------- Output ----------------
        self.output_video_path = "output_video.mp4"
        self.distance_log_file = "distance_log.txt"

        # ---------------- Runtime Settings ----------------
        self.max_reconnect_attempts = 200000
        self.reconnect_delay = 1

        self.is_prerecorded_video = False
        self.reconnect_attempts = 0

        # ---------------- Speed State ----------------
        self.skip_first_measurement = True
        self.rotation_time_seconds = 0.0
        self.machine_speed_bph = 0.0

        self._initialize_models()

    # =========================================================
    # File Utilities
    # =========================================================
    def write_text_file(self, text: str, path: str = "speed.txt"):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        with open(path, "w", buffering=1) as f:
            f.write(f"{text}\n")
            f.flush()

    # =========================================================
    # Video Stream
    # =========================================================
    def open_video_stream(self):

        cap = cv2.VideoCapture(self.input_video_path)
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

        if not cap.isOpened():
            return None

        return cap

    # =========================================================
    # Model Initialization
    # =========================================================
    def _initialize_models(self):

        self.is_prerecorded_video = ".mp4" in self.input_video_path
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.marker_detector = YOLO(self.marker_detection_model_path).to(self.device).eval()
        self.drum_segmenter = YOLO(self.drum_segmentation_model_path).to(self.device).eval()

    # =========================================================
    # Mask Utilities
    # =========================================================
    def tensor_mask_to_uint8(self, mask_tensor: torch.Tensor):

        mask = (mask_tensor * 255).to(dtype=torch.uint8)
        return mask.cpu().numpy()

    # =========================================================
    # Geometry Utilities
    # =========================================================
    def get_point_side_of_line(self, point, line_start, line_end):
        """
        Returns:
        +1 → point on left side
        -1 → point on right side
        """

        px, py = point
        x1, y1 = line_start
        x2, y2 = line_end

        cross = (x2 - x1) * (py - y1) - (y2 - y1) * (px - x1)

        return 1 if cross > 0 else -1

    # =========================================================
    # Main Speed Detection
    # =========================================================
    def run_speed_monitor(self):

        cap = self.open_video_stream()

        while cap is None:
            print(f"Stream connection failed. Retrying in {self.reconnect_delay}s")

            self.reconnect_attempts += 1

            if self.reconnect_attempts >= self.max_reconnect_attempts:
                self.reconnect_attempts = 0

            time.sleep(self.reconnect_delay)
            cap = self.open_video_stream()

        self.reconnect_attempts = 0

        # -----------------------------------------------------
        # Read First Frame
        # -----------------------------------------------------
        success, first_frame = cap.read()

        if not success:
            raise RuntimeError("Unable to read video.")

        with torch.inference_mode():
            segmentation_result = self.drum_segmenter(first_frame)[0]

        if segmentation_result.masks is None:
            raise RuntimeError("No drum mask detected.")

        drum_mask = self.tensor_mask_to_uint8(segmentation_result.masks.data[0])

        self.video_fps = int(cap.get(cv2.CAP_PROP_FPS))

        resized_mask = cv2.resize(
            drum_mask,
            (first_frame.shape[1], first_frame.shape[0])
        )

        # -----------------------------------------------------
        # Calculate Drum Pivot
        # -----------------------------------------------------
        moments = cv2.moments(resized_mask, binaryImage=True)

        self.drum_pivot = (
            int(moments["m10"] / moments["m00"]),
            int(moments["m01"] / moments["m00"])
        )

        mask_boolean = resized_mask > 0

        # -----------------------------------------------------
        # Create Reference Line (2-Quadrant Divider)
        # -----------------------------------------------------
        reference_angle_deg = 60
        reference_angle_rad = np.radians(reference_angle_deg)

        line_length = 130

        dx = int(line_length * np.cos(reference_angle_rad))
        dy = int(line_length * np.sin(reference_angle_rad))

        line_start = (self.drum_pivot[0] - dx, self.drum_pivot[1] - dy)
        line_end = (self.drum_pivot[0] + dx, self.drum_pivot[1] + dy)

        # -----------------------------------------------------
        # Video Writer
        # -----------------------------------------------------
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        fourcc = cv2.VideoWriter_fourcc(*"mp4v")

        video_writer = cv2.VideoWriter(
            self.output_video_path,
            fourcc,
            self.video_fps,
            (width, height),
        )

        # -----------------------------------------------------
        # Tracking State
        # -----------------------------------------------------
        previous_side = None
        current_side = None

        last_cross_frame = 0
        frame_index = 1

        previous_x = None
        previous_y = None

        # -----------------------------------------------------
        # Overlay Layout
        # -----------------------------------------------------
        TEXT_X = 60
        TEXT_Y = 220
        LINE_GAP = 80

        TEXT_SCALE = 0.9
        TEXT_THICKNESS = 2

        # =====================================================
        # Frame Loop
        # =====================================================
        while True:

            frame_start_time = time.time()

            success, frame = cap.read()

            if not success:

                if self.is_prerecorded_video:
                    break

                print("Frame read failed. Attempting reconnect...")

                self.reconnect_attempts += 1

                cap.release()
                time.sleep(self.reconnect_delay)

                cap = self.open_video_stream()

                if cap is None or self.reconnect_attempts >= self.max_reconnect_attempts:
                    print("Max retries reached.")
                    break

                continue

            self.reconnect_attempts = 0

            display_frame = frame.copy()
            detection_frame = frame.copy()

            # -------------------------------------------------
            # Draw Reference Line + Pivot
            # -------------------------------------------------
            cv2.line(display_frame, line_start, line_end, (255, 0, 255), 2)

            cv2.circle(display_frame, self.drum_pivot, 6, (0, 255, 255), -1)

            cv2.putText(
                display_frame,
                "Pivot",
                (self.drum_pivot[0] + 10, self.drum_pivot[1]),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 255),
                2
            )

            # -------------------------------------------------
            # Marker Detection
            # -------------------------------------------------
            detection_result = self.marker_detector(detection_frame)[0]

            boxes = getattr(detection_result, "boxes", None)

            best_detection = None

            if boxes is not None:

                for box in boxes:

                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    cx = (x1 + x2) // 2
                    cy = (y1 + y2) // 2

                    if 0 <= cx < width and 0 <= cy < height and mask_boolean[cy, cx]:

                        confidence = float(box.conf)

                        if best_detection is None or confidence > best_detection[0]:

                            best_detection = (
                                confidence,
                                cx,
                                cy,
                                x1,
                                y1,
                                x2,
                                y2
                            )

            # -------------------------------------------------
            # Process Detection
            # -------------------------------------------------
            if best_detection:

                conf, cx, cy, x1, y1, x2, y2 = best_detection

                cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 210, 0), 2)

                cv2.circle(display_frame, (cx, cy), 5, (0, 0, 255), -1)

                if previous_x is None:
                    previous_x = cx
                    previous_y = cy

                movement_pixels = np.hypot(cx - previous_x, cy - previous_y)

                previous_x = cx
                previous_y = cy

                # Distance logging
                with open(self.distance_log_file, "a") as log_file:

                    log_file.write(
                        f"second:{frame_index/self.video_fps} distance:{movement_pixels}\n"
                    )

                current_side = self.get_point_side_of_line(
                    (cx, cy),
                    line_start,
                    line_end
                )

                # -------------------------------------------------
                # Initialize first side
                # -------------------------------------------------
                if previous_side is None:

                    previous_side = current_side
                    last_cross_frame = frame_index

                # -------------------------------------------------
                # Detect Line Crossing
                # -------------------------------------------------
                elif (
                    current_side != previous_side
                    and movement_pixels <= 100
                ):

                    delta_time = (frame_index - last_cross_frame) / self.video_fps

                    estimated_rotation_time = delta_time * 2

                    if self.skip_first_measurement:

                        self.skip_first_measurement = False
                        last_cross_frame = frame_index
                        previous_side = current_side
                        continue

                    if estimated_rotation_time >= 1.5:

                        last_cross_frame = frame_index
                        previous_side = current_side

                        self.rotation_time_seconds = round(estimated_rotation_time, 3)

                        with open("speed.txt", "w") as f:
                            f.write(str(self.rotation_time_seconds))

                        if self.rotation_time_seconds != 0:

                            self.machine_speed_bph = round(
                                (12 * 3600) / self.rotation_time_seconds,
                                3
                            )

                        print(self.rotation_time_seconds)

                        yield self.rotation_time_seconds

            frame_index += 1

            # -------------------------------------------------
            # Overlays
            # -------------------------------------------------
            cv2.putText(
                display_frame,
                f"Est. Rotation Time: {self.rotation_time_seconds:.2f}s",
                (TEXT_X, TEXT_Y),
                cv2.FONT_HERSHEY_SIMPLEX,
                TEXT_SCALE,
                (0, 255, 255),
                TEXT_THICKNESS
            )

            if self.machine_speed_bph > 0:

                speed_string = f"{int(round(self.machine_speed_bph)):,}"

                cv2.putText(
                    display_frame,
                    f"Speed: {speed_string} btl./hr",
                    (TEXT_X, TEXT_Y + LINE_GAP),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    TEXT_SCALE,
                    (0, 255, 0),
                    TEXT_THICKNESS
                )

            else:

                cv2.putText(
                    display_frame,
                    "Speed: --",
                    (TEXT_X, TEXT_Y + LINE_GAP),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    TEXT_SCALE,
                    (0, 255, 0),
                    TEXT_THICKNESS
                )

            frame_time = time.time() - frame_start_time
            print("time per frame:", frame_time)

            video_writer.write(display_frame)

        cap.release()
        video_writer.release()

        print(f"Saved output video → {self.output_video_path}")


# =============================================================
# Run
# =============================================================
monitor = SpeedMonitor()

for rotation_time in monitor.run_speed_monitor():
    pass

time per frame :  0.05892348289489746
time per frame :  0.01409912109375
time per frame :  0.013996124267578125
time per frame :  0.013519048690795898
time per frame :  0.012928247451782227
time per frame :  0.012943506240844727
time per frame :  0.01144099235534668
time per frame :  0.011059999465942383
time per frame :  0.011201858520507812
time per frame :  0.011571645736694336
time per frame :  0.011932849884033203
time per frame :  0.013523340225219727
time per frame :  0.011368513107299805
time per frame :  0.012825489044189453
time per frame :  0.013204336166381836
time per frame :  0.011205673217773438
time per frame :  0.01109170913696289
time per frame :  0.010907173156738281
time per frame :  0.010919570922851562
time per frame :  0.010842561721801758
time per frame :  0.010834693908691406
time per frame :  0.010817527770996094
time per frame :  0.010806083679199219
time per frame :  0.010743141174316406
time per frame :  0.012790441513061523
time per frame :  0.011123180389

In [ ]:
from google.colab import files


files.download("output_video.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>